In [ ]:
import os
import re
import glob
import dustpy.constants as c
from dustpy.utils import read_data
import h5py
import numpy as np
import matplotlib.pyplot as plt
from astropy import constants as apc

In [ ]:
###plotting the upper left panel from the ipanel in DustPy###

# set-up the framework
DATA_DIR = 'data'          
FILENAME = 'data'          
EXTENSION = 'hdf5'

SNAPSHOTS = [10, 15, 21]   # corresponds to: data0010, data0015, data0021

SAVE_FIGURES = True
OUTDIR = 'plots'

SHOW_LIMITS = True         # show drift- and fragmentationlimit
SHOW_ST1 = True            # show stokes St = 1 line



# read data
data = read_data(DATA_DIR, filename=FILENAME, extension=EXTENSION)

# declare file numbers as internal indices
files = sorted(glob.glob(os.path.join(DATA_DIR, f"{FILENAME}*.{EXTENSION}")))

file_numbers = []
for file in files:
    match = re.search(rf"{FILENAME}(\d+)\.{EXTENSION}$", os.path.basename(file))
    if match is not None:
        file_numbers.append(int(match.group(1)))

snapshot_to_index = {number: index for index, number in enumerate(file_numbers)}

missing = [snap for snap in SNAPSHOTS if snap not in snapshot_to_index]
if missing:
    raise FileNotFoundError(
        f'files not found: '
        + ','.join([f"{FILENAME}{snap:04d}.{EXTENSION}" for snap in missing])
    )

if SAVE_FIGURES:
    os.makedirs(OUTDIR, exist_ok=True)


# ploting function
def plot_ax00(snapshot_number):
    it = snapshot_to_index[snapshot_number]

    # same level defintions as in DustPy-panel/ipanel
    sd_max = np.ceil(np.log10(np.nanmax(data.dust.sigma)))
    levels = np.linspace(sd_max - 6, sd_max, 7)

    fig, ax = plt.subplots(figsize=(6.0, 4.6), dpi=200)

    # protection against log10(0)
    sigma = np.clip(data.dust.sigma[it, ...], 10.0**levels[0], None)

    # dust surface density distribution
    im = ax.contourf(data.grid.r[it, ...] / c.au, data.grid.m[it, ...], np.log10(sigma.T), levels=levels, cmap="magma", extend="both")

    # St = 1 line
    if SHOW_ST1:
        ax.contour(data.grid.r[it, ...] / c.au, data.grid.m[it, ...], data.dust.St[it, ...].T, levels=[1.0], colors="white", linewidths=1.5)

    # drift- and fragmentationlimit
    if SHOW_LIMITS:
        ax.contour(data.grid.r[it, ...] / c.au,data.grid.m[it, ...],(data.dust.St - data.dust.St_limits.drift[..., None])[it, ...].T,levels=[0.0],colors="C2",linewidths=1.2)

        ax.contour(data.grid.r[it, ...] / c.au, data.grid.m[it, ...], (data.dust.St - data.dust.St_limits.frag[..., None])[it, ...].T, levels=[0.0], colors="C0", linewidths=1.2)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(data.grid.r[it, 0] / c.au, data.grid.r[it, -1] / c.au)
    ax.set_ylim(data.grid.m[it, 0], data.grid.m[it, -1])
    ax.set_xlabel("Distance from star [AU]")
    ax.set_ylabel("Particle mass [g]")
    ax.set_title(rf"data{snapshot_number:04d}, $t = {data.t[it] / c.year:.2e}$ yr")

    # colorbar
    cbar = fig.colorbar(im, ax=ax, pad=0.02)
    cbar.ax.set_ylabel(r"$\sigma_\mathrm{d}$ [g cm$^{-2}$]")
    cbar.set_ticks(levels)
    cbar.set_ticklabels([rf"$10^{{{int(level)}}}$" for level in levels])

    fig.tight_layout()

    if SAVE_FIGURES:
        pdf_path = os.path.join(OUTDIR, f"ax00_data{snapshot_number:04d}.pdf")
        png_path = os.path.join(OUTDIR, f"ax00_data{snapshot_number:04d}.png")

        fig.savefig(pdf_path, bbox_inches="tight")
        fig.savefig(png_path, bbox_inches="tight", dpi=300)

        print(f"Saved: {pdf_path}")
        print(f"Saved: {png_path}")

    plt.show()

    return fig, ax



# produce plots
for snap in SNAPSHOTS:
    plot_ax00(snap)

In [ ]:
print('#########################################################')
print('modified dustPy, 7.943e3 yr')
print('#########################################################')


with h5py.File('data/data0010.hdf5','r') as f:
    #print(list(f.keys()))


    r = f['grid/r']
    r = np.asarray(r/apc.au.cgs.value)
    #print(f'r [au]:{r}')


    Sigmad = f['dust/Sigma']
    Sigmad = np.asarray(Sigmad)
    #print(f'Sigma_d [cm^2 g^-1]:{Sigmad}')
    Sigmag = f['gas/Sigma']
    Sigmag = np.asarray(Sigmag)
    #print(f'Sigma_g [cm^2 g^-1]:{Sigmag}')


    Flux_adv = f['dust/Fi/adv']
    Flux_adv = np.asarray(Flux_adv)
    #print(f'F_adv:{Flux_adv}')
    Flux_diff = f['dust/Fi/diff']
    Flux_diff = np.asarray(Flux_diff)
    #print(f'F_diff:{Flux_diff}')
    Flux_tot = f['dust/Fi/tot']
    Flux_tot = np.asarray(Flux_tot)
    #print(f'F_tot:{Flux_tot}')

    v_brown = f['dust/v/rel/brown']
    v_brown = np.asarray(v_brown)
    #print(f'v_brown:{v_brown}')
    v_rel_tot = f['dust/v/rel/tot']
    v_rel_tot = np.asarray(v_rel_tot)
    #print(f'v_rel_tot:{v_rel_tot}')

    eps = f['dust/eps']
    eps = np.asarray(eps)
    #print(f'eps:{eps}')

    coag = f['dust/S/coag']
    coag = np.asarray(coag)
    #print(f'coagulation: {coag}')

    p_frag = f['dust/p/frag']
    p_stick = f['dust/p/stick']
    p_frag = np.asarray(p_frag)
    p_stick = np.asarray(p_stick)


    delta_rad = f['dust/delta/rad']
    delta_rad = np.asarray(delta_rad)
    #print(f'Delta_rad:{delta_rad}')
    delta_turb = f['dust/delta/turb']
    delta_turb = np.asarray(delta_turb)
    #print(f'Delta_turb:{delta_turb}')
    delta_vert = f['dust/delta/vert']
    delta_vert = np.asarray(delta_vert)
    #print(f'Delta_vert:{delta_vert}')

print('#########################################################')
print('vanilla dustPy, 7.943e3 yr')
print('#########################################################')

with h5py.File('data/data_nat/data0010.hdf5','r') as f:
    #print(list(f.keys()))


    r_nat = f['grid/r']
    r_nat = np.asarray(r_nat/apc.au.cgs.value)
    #print(f'r [au]:{r_nat}')


    Sigmad_nat = f['dust/Sigma']
    Sigmad_nat = np.asarray(Sigmad_nat)
    #print(f'Sigma_d [cm^2 g^-1]:{Sigmad_nat}')
    Sigmag_nat = f['gas/Sigma']
    Sigmag_nat = np.asarray(Sigmag_nat)
    #print(f'Sigma_g [cm^2 g^-1]:{Sigmag_nat}')


    Flux_adv_nat = f['dust/Fi/adv']
    Flux_adv_nat = np.asarray(Flux_adv_nat)
    #print(f'F_adv:{Flux_adv_nat}')
    Flux_diff_nat = f['dust/Fi/diff']
    Flux_diff_nat = np.asarray(Flux_diff_nat)
    #print(f'F_diff:{Flux_diff_nat}')
    Flux_tot_nat = f['dust/Fi/tot']
    Flux_tot_nat = np.asarray(Flux_tot_nat)
    #print(f'F_tot:{Flux_tot_nat}')

    v_brown_nat = f['dust/v/rel/brown']
    v_brown_nat = np.asarray(v_brown_nat)
    #print(f'v_brown:{v_brown_nat}')
    v_rel_tot_nat = f['dust/v/rel/tot']
    v_rel_tot_nat = np.asarray(v_rel_tot_nat)
    #print(f'v_rel_tot:{v_rel_tot_nat}')

    eps_nat = f['dust/eps']
    eps_nat = np.asarray(eps_nat)
    #print(f'eps:{eps_nat}')

    coag_nat = f['dust/S/coag']
    coag_nat = np.asarray(coag_nat)
    #print(f'coagulation: {coag_nat}')

    p_frag_nat = f['dust/p/frag']
    p_stick_nat = f['dust/p/stick']
    p_frag_nat = np.asarray(p_frag_nat)
    p_stick_nat = np.asarray(p_stick_nat)

    delta_rad_nat = f['dust/delta/rad']
    delta_rad_nat = np.asarray(delta_rad_nat)
    #print(f'Delta_rad:{delta_rad_nat}')
    delta_turb_nat = f['dust/delta/turb']
    delta_turb_nat = np.asarray(delta_turb_nat)
    #print(f'Delta_turb:{delta_turb_nat}')
    delta_vert_nat = f['dust/delta/vert']
    delta_vert_nat = np.asarray(delta_vert_nat)
    #print(f'Delta_vert:{delta_vert_nat}')
      
print('#########################################################')
print('comparison , 7.943e3 yr')
print('#########################################################')

delta_Sigmad = Sigmad - Sigmad_nat
#print(f'change in Sigma_d:{delta_Sigmad}')
delta_Sigmag = Sigmag - Sigmag_nat
#print(f'change in Sigma_g:{delta_Sigmag}')

delta_flux_adv = Flux_adv - Flux_adv_nat
#print(f'change in F_adv: {delta_flux_adv}')
delta_flux_diff = Flux_diff - Flux_diff_nat
#print(f'change in F_diff: {delta_flux_diff}')
delta_flux_tot = Flux_tot - Flux_tot_nat
#print(f'change in F_tot: {delta_flux_tot}')

delta_v_brown = v_brown - v_brown_nat
#print(f'change in v_brown:{delta_v_brown}')
delta_v_rel_tot = v_rel_tot - v_rel_tot_nat
#print(f'change in v_rel_tot:{delta_v_rel_tot}')

delta_eps = eps - eps_nat
#print(f'change in epsilon:{delta_eps}')

delta_coag = coag - coag_nat
#print(f'change in coagulation:{delta_coag}')


with h5py.File('data/data0010.hdf5','r') as f:
    ri = np.asarray(f['grid/ri'])
    ri = ri / apc.au.cgs.value

Flux_adv = np.sum(Flux_adv, axis = 1)
Flux_adv_nat = np.sum(Flux_adv_nat, axis = 1)

fig, ax0 = plt.subplots(1, 1, dpi=300)
ax0.plot(ri, Flux_adv_nat, '--',color = 'grey',  alpha = 0.5, label = r'$F_{\mathrm{adv,vanilla}}$' )
ax0.plot(ri, Flux_adv,color = 'green',  label = r'$F_{\mathrm{adv, update}}$')
ax0.grid(True, which = 'both', alpha = 0.2)
ax0.set_xscale('log')
#ax0.set_yscale('log')
ax0.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax0.set_ylabel(r'$F_{\mathrm{adv}}$')
ax0.set_xlim(1,900)
ax0.set_title(r'Advection flux after $7.943\times 10^{3}$years')
ax0.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/flux_adv_data0010.png', dpi=300, bbox_inches="tight")
plt.show

Flux_diff = np.sum(Flux_diff, axis = 1)
Flux_diff_nat = np.sum(Flux_diff_nat, axis = 1)

fig, ax1 = plt.subplots(1, 1, dpi=300)
ax1.plot(ri, Flux_diff_nat, '--',color = 'grey',  alpha = 0.5, label = r'$F_{\mathrm{diff,vanilla}}$' )
ax1.plot(ri, Flux_diff,color = 'green',  label = r'$F_{\mathrm{diff, update}}$')
ax1.grid(True, which = 'both', alpha = 0.2)
ax1.set_xscale('log')
#ax1.set_yscale('log')
ax1.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax1.set_ylabel(r'$F_{\mathrm{diff}}$')
ax1.set_xlim(1,900)
ax1.set_title(r'Diffusion flux after $7.943\times 10^{3}$years')
ax1.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/flux_diff_data0010.png', dpi=300, bbox_inches="tight")
plt.show

Flux_tot = np.sum(Flux_tot, axis = 1)
Flux_tot_nat = np.sum(Flux_tot_nat, axis = 1)

fig, ax2 = plt.subplots(1, 1, dpi=300)
ax2.plot(ri, Flux_tot_nat, '--',color = 'grey',  alpha = 0.5, label = r'$F_{\mathrm{tot,vanilla}}$' )
ax2.plot(ri, Flux_tot,color = 'green',  label = r'$F_{\mathrm{tot, update}}$')
ax2.grid(True, which = 'both', alpha = 0.2)
ax2.set_xscale('log')
#ax2.set_yscale('log')
ax2.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax2.set_ylabel(r'$F_{\mathrm{tot}}$')
ax2.set_xlim(1,900)
ax2.set_title(r'Total flux after $7.943\times 10^{3}$years')
ax2.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/flux_tot_data0010.png', dpi=300, bbox_inches="tight")
plt.show


#surface denisities
Sigmad = np.sum(Sigmad, axis = 1)
Sigmad_nat = np.sum(Sigmad_nat, axis = 1)

fig, ax3 = plt.subplots(1, 1, dpi=300)
ax3.plot(r, Sigmad_nat, '--',color = 'grey',  alpha = 0.5, label = r'$\Sigma_{\mathrm{d,vanilla}}$' )
ax3.plot(r, Sigmad,color = 'green',  label = r'$\Sigma_{\mathrm{d, update}}$')
ax3.grid(True, which = 'both', alpha = 0.2)
ax3.set_xscale('log')
#ax3.set_yscale('log')
ax3.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax3.set_ylabel(r'$\Sigma_{\mathrm{d}}\ [g\ cm^{-2}]$')
ax3.set_xlim(1,900)
ax3.set_title(r'Dust surface density after $7.943\times 10^{3}$years')
ax3.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/Sigmad_data0010.png', dpi=300, bbox_inches="tight")
plt.show

#Sigmag = np.sum(Sigmag, axis = 1)
#Sigmag_nat = np.sum(Sigmag_nat, axis = 1)

fig, ax4 = plt.subplots(1, 1, dpi=300)
ax4.plot(r, Sigmag_nat, '--',color = 'grey',  alpha = 0.5, label = r'$\Sigma_{\mathrm{g,vanilla}}$' )
ax4.plot(r, Sigmag,color = 'green',  label = r'$\Sigma_{\mathrm{g, update}}$')
ax4.grid(True, which = 'both', alpha = 0.2)
ax4.set_xscale('log')
#ax4.set_yscale('log')
ax4.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax4.set_ylabel(r'$\Sigma_{\mathrm{g}}\ [g\ cm^{-2}]$')
ax4.set_xlim(1,900)
ax4.set_title(r'Gas surface density after $7.943\times 10^{3}$years')
ax4.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/Sigmag_data0010.png', dpi=300, bbox_inches="tight")
plt.show

#dust to gas ratio
fig, ax5 = plt.subplots(1, 1, dpi=300)
ax5.plot(r, eps_nat, '--',color = 'grey',  alpha = 0.5, label = r'$\epsilon_{\mathrm{vanilla}}$' )
ax5.plot(r, eps,color = 'green',  label = r'$\epsilon_{\mathrm{update}}$')
ax5.grid(True, which = 'both', alpha = 0.2)
ax5.set_xscale('log')
ax5.set_yscale('log')
ax5.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax5.set_ylabel(r'$\epsilon$')
ax5.set_xlim(1,900)
ax5.set_title(r'Dust to gas ratio after $7.943\times 10^{3}$years')
ax5.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/epsilon_data0010.png', dpi=300, bbox_inches="tight")
plt.show

#relative velocities
v_brown = np.sum(v_brown, axis = 1)
v_brown = np.sum(v_brown, axis = 1)
v_brown_nat = np.sum(v_brown_nat, axis = 1)
v_brown_nat = np.sum(v_brown_nat, axis = 1)

fig, ax6 = plt.subplots(1, 1, dpi=300)
ax6.plot(r, v_brown_nat, '--',color = 'grey',  alpha = 0.5, label = r'$v_{\mathrm{brown,vanilla}}$' )
ax6.plot(r, v_brown,color = 'green',  label = r'$v_{\mathrm{brown, update}}$')
ax6.grid(True, which = 'both', alpha = 0.2)
ax6.set_xscale('log')
ax6.set_yscale('log')
ax6.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax6.set_ylabel(r'$v_{\mathrm{brown}}\ [cm\ s^{-1}]$')
ax6.set_xlim(1,900)
ax6.set_title(r'Velocity after $7.943\times 10^{3}$years')
fig.savefig('pictures/v_brown_data0010.png', dpi=300, bbox_inches="tight")
ax6.legend(loc = 'best')

plt.tight_layout()
plt.show

v_rel_tot = np.sum(v_rel_tot, axis = 1)
v_rel_tot = np.sum(v_rel_tot, axis = 1)
v_rel_tot_nat = np.sum(v_rel_tot_nat, axis = 1)
v_rel_tot_nat = np.sum(v_rel_tot_nat, axis = 1)

fig, ax7 = plt.subplots(1, 1, dpi=300)
ax7.plot(r, v_rel_tot_nat, '--',color = 'grey',  alpha = 0.5, label = r'$v_{\mathrm{rel, tot, ,vanilla}}$' )
ax7.plot(r, v_rel_tot,color = 'green',  label = r'$v_{\mathrm{rel, tot, update}}$')
ax7.grid(True, which = 'both', alpha = 0.2)
ax7.set_xscale('log')
ax7.set_yscale('log')
ax7.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax7.set_ylabel(r'$v_{\mathrm{rel, tot}}\ [cm\ s^{-1}]$')
ax7.set_xlim(1,900)
ax7.set_title(r'Velocity after $7.943\times 10^{3}$years')
ax7.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/v_rel_tot_data0010.png', dpi=300, bbox_inches="tight")
plt.show

#fragmentation probabilities
p_frag = np.sum(p_frag, axis = 1)
p_frag = np.sum(p_frag, axis = 1)
p_frag_nat = np.sum(p_frag_nat, axis = 1)
p_frag_nat = np.sum(p_frag_nat, axis = 1)

fig, ax8 = plt.subplots(1, 1, dpi=300)
ax8.plot(r, p_frag_nat, '--',color = 'grey',  alpha = 0.5, label = r'$p_{\mathrm{frag, vanilla}}$' )
ax8.plot(r, p_frag,color = 'green',  label = r'$p_{\mathrm{frag, update}}$')
ax8.grid(True, which = 'both', alpha = 0.2)
ax8.set_xscale('log')
#ax8.set_yscale('log')
ax8.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax8.set_ylabel(r'$p_{\mathrm{frag}}$')
ax8.set_xlim(1,900)
ax8.set_title(r'Fragmentation probability $7.943\times 10^{3}$years')
ax8.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/p_frag_data0010.png', dpi=300, bbox_inches="tight")
plt.show

p_stick = np.sum(p_stick, axis = 1)
p_stick = np.sum(p_stick, axis = 1)
p_stick_nat = np.sum(p_stick_nat, axis = 1)
p_stick_nat = np.sum(p_stick_nat, axis = 1)

fig, ax9 = plt.subplots(1, 1, dpi=300)
ax9.plot(r, p_stick_nat, '--',color = 'grey',  alpha = 0.5, label = r'$p_{\mathrm{stick, vanilla}}$' )
ax9.plot(r, p_stick, color = 'green',  label = r'$p_{\mathrm{stick, update}}$')
ax9.grid(True, which = 'both', alpha = 0.2)
ax9.set_xscale('log')
#ax9.set_yscale('log')
ax9.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax9.set_ylabel(r'$p_{\mathrm{stick}}$')
ax9.set_xlim(1,900)
ax9.set_title(r'Sticking probability $7.943\times 10^{3}$years')
ax9.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/p_stick_data0010.png', dpi=300, bbox_inches="tight")
plt.show



In [ ]:
print('#########################################################')
print('modified dustPy, 2.512e4 yr')
print('#########################################################')


with h5py.File('data/data0015.hdf5','r') as f:
    #print(list(f.keys()))


    r = f['grid/r']
    r = np.asarray(r/apc.au.cgs.value)
    #print(f'r [au]:{r}')


    Sigmad = f['dust/Sigma']
    Sigmad = np.asarray(Sigmad)
    #print(f'Sigma_d [cm^2 g^-1]:{Sigmad}')
    Sigmag = f['gas/Sigma']
    Sigmag = np.asarray(Sigmag)
    #print(f'Sigma_g [cm^2 g^-1]:{Sigmag}')


    Flux_adv = f['dust/Fi/adv']
    Flux_adv = np.asarray(Flux_adv)
    #print(f'F_adv:{Flux_adv}')
    Flux_diff = f['dust/Fi/diff']
    Flux_diff = np.asarray(Flux_diff)
    #print(f'F_diff:{Flux_diff}')
    Flux_tot = f['dust/Fi/tot']
    Flux_tot = np.asarray(Flux_tot)
    #print(f'F_tot:{Flux_tot}')

    v_brown = f['dust/v/rel/brown']
    v_brown = np.asarray(v_brown)
    #print(f'v_brown:{v_brown}')
    v_rel_tot = f['dust/v/rel/tot']
    v_rel_tot = np.asarray(v_rel_tot)
    #print(f'v_rel_tot:{v_rel_tot}')

    eps = f['dust/eps']
    eps = np.asarray(eps)
    #print(f'eps:{eps}')

    coag = f['dust/S/coag']
    coag = np.asarray(coag)
    #print(f'coagulation: {coag}')

    p_frag = f['dust/p/frag']
    p_stick = f['dust/p/stick']
    p_frag = np.asarray(p_frag)
    p_stick = np.asarray(p_stick)

    delta_rad = f['dust/delta/rad']
    delta_rad = np.asarray(delta_rad)
    #print(f'Delta_rad:{delta_rad}')
    delta_turb = f['dust/delta/turb']
    delta_turb = np.asarray(delta_turb)
    #print(f'Delta_turb:{delta_turb}')
    delta_vert = f['dust/delta/vert']
    delta_vert = np.asarray(delta_vert)
    #print(f'Delta_vert:{delta_vert}')

print('#########################################################')
print('vanilla dustPy, 2.512e4 yr')
print('#########################################################')

with h5py.File('data/data_nat/data0010.hdf5','r') as f:
    #print(list(f.keys()))


    r_nat = f['grid/r']
    r_nat = np.asarray(r_nat/apc.au.cgs.value)
    #print(f'r [au]:{r_nat}')


    Sigmad_nat = f['dust/Sigma']
    Sigmad_nat = np.asarray(Sigmad_nat)
    #print(f'Sigma_d [cm^2 g^-1]:{Sigmad_nat}')
    Sigmag_nat = f['gas/Sigma']
    Sigmag_nat = np.asarray(Sigmag_nat)
    #print(f'Sigma_g [cm^2 g^-1]:{Sigmag_nat}')


    Flux_adv_nat = f['dust/Fi/adv']
    Flux_adv_nat = np.asarray(Flux_adv_nat)
    #print(f'F_adv:{Flux_adv_nat}')
    Flux_diff_nat = f['dust/Fi/diff']
    Flux_diff_nat = np.asarray(Flux_diff_nat)
    #print(f'F_diff:{Flux_diff_nat}')
    Flux_tot_nat = f['dust/Fi/tot']
    Flux_tot_nat = np.asarray(Flux_tot_nat)
    #print(f'F_tot:{Flux_tot_nat}')

    v_brown_nat = f['dust/v/rel/brown']
    v_brown_nat = np.asarray(v_brown_nat)
    #print(f'v_brown:{v_brown_nat}')
    v_rel_tot_nat = f['dust/v/rel/tot']
    v_rel_tot_nat = np.asarray(v_rel_tot_nat)
    #print(f'v_rel_tot:{v_rel_tot_nat}')

    eps_nat = f['dust/eps']
    eps_nat = np.asarray(eps_nat)
    #print(f'eps:{eps_nat}')

    coag_nat = f['dust/S/coag']
    coag_nat = np.asarray(coag_nat)
    #print(f'coagulation: {coag_nat}')

    p_frag_nat = f['dust/p/frag']
    p_stick_nat = f['dust/p/stick']
    p_frag_nat = np.asarray(p_frag_nat)
    p_stick_nat = np.asarray(p_stick_nat)

    delta_rad_nat = f['dust/delta/rad']
    delta_rad_nat = np.asarray(delta_rad_nat)
    #print(f'Delta_rad:{delta_rad_nat}')
    delta_turb_nat = f['dust/delta/turb']
    delta_turb_nat = np.asarray(delta_turb_nat)
    #print(f'Delta_turb:{delta_turb_nat}')
    delta_vert_nat = f['dust/delta/vert']
    delta_vert_nat = np.asarray(delta_vert_nat)
    #print(f'Delta_vert:{delta_vert_nat}')
      
print('#########################################################')
print('comparison , 2.512e4 yr')
print('#########################################################')

delta_Sigmad = Sigmad - Sigmad_nat
print(f'change in Sigma_d:{delta_Sigmad}')
delta_Sigmag = Sigmag - Sigmag_nat
print(f'change in Sigma_g:{delta_Sigmag}')

delta_flux_adv = Flux_adv - Flux_adv_nat
print(f'change in F_adv: {delta_flux_adv}')
delta_flux_diff = Flux_diff - Flux_diff_nat
print(f'change in F_diff: {delta_flux_diff}')
delta_flux_tot = Flux_tot - Flux_tot_nat
print(f'change in F_tot: {delta_flux_tot}')

delta_v_brown = v_brown - v_brown_nat
print(f'change in v_brown:{delta_v_brown}')
delta_v_rel_tot = v_rel_tot - v_rel_tot_nat
print(f'change in v_rel_tot:{delta_v_rel_tot}')

delta_eps = eps - eps_nat
print(f'change in epsilon:{delta_eps}')

delta_coag = coag - coag_nat
print(f'change in coagulation:{delta_coag}')



with h5py.File('data/data0015.hdf5','r') as f:
    ri = np.asarray(f['grid/ri'])
    ri = ri / apc.au.cgs.value

Flux_adv = np.sum(Flux_adv, axis = 1)
Flux_adv_nat = np.sum(Flux_adv_nat, axis = 1)


fig, ax0 = plt.subplots(1, 1, dpi=300)
ax0.plot(ri, Flux_adv_nat, '--',color = 'grey',  alpha = 0.5, label = r'$F_{\mathrm{adv,vanilla}}$' )
ax0.plot(ri, Flux_adv,color = 'green',  label = r'$F_{\mathrm{adv, update}}$')
ax0.grid(True, which = 'both', alpha = 0.2)
ax0.set_xscale('log')
#ax0.set_yscale('log')
ax0.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax0.set_ylabel(r'$F_{\mathrm{adv}}$')
ax0.set_xlim(1,900)
ax0.set_title(r'Advection flux after $7.943 10^{3}$years')
ax0.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/flux_adv_data0015.png', dpi=300, bbox_inches="tight")
plt.show

Flux_diff = np.sum(Flux_diff, axis = 1)
Flux_diff_nat = np.sum(Flux_diff_nat, axis = 1)

fig, ax1 = plt.subplots(1, 1, dpi=300)
ax1.plot(ri, Flux_diff_nat, '--',color = 'grey',  alpha = 0.5, label = r'$F_{\mathrm{diff,vanilla}}$' )
ax1.plot(ri, Flux_diff,color = 'green',  label = r'$F_{\mathrm{diff, update}}$')
ax1.grid(True, which = 'both', alpha = 0.2)
ax1.set_xscale('log')
#ax1.set_yscale('log')
ax1.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax1.set_ylabel(r'$F_{\mathrm{diff}}$')
ax1.set_xlim(1,900)
ax1.set_title(r'Diffusion flux after $7.943 10^{3}$years')
ax1.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/flux_diff_data0015.png', dpi=300, bbox_inches="tight")
plt.show

Flux_tot = np.sum(Flux_tot, axis = 1)
Flux_tot_nat = np.sum(Flux_tot_nat, axis = 1)

fig, ax2 = plt.subplots(1, 1, dpi=300)
ax2.plot(ri, Flux_tot_nat, '--',color = 'grey',  alpha = 0.5, label = r'$F_{\mathrm{tot,vanilla}}$' )
ax2.plot(ri, Flux_tot,color = 'green',  label = r'$F_{\mathrm{tot, update}}$')
ax2.grid(True, which = 'both', alpha = 0.2)
ax2.set_xscale('log')
#ax2.set_yscale('log')
ax2.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax2.set_ylabel(r'$F_{\mathrm{tot}}$')
ax2.set_xlim(1,900)
ax2.set_title(r'Total flux after $2.512\times 10^{4}$years')
ax2.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/flux_tot_data0015.png', dpi=300, bbox_inches="tight")
plt.show

#surface denisities
Sigmad = np.sum(Sigmad, axis = 1)
Sigmad_nat = np.sum(Sigmad_nat, axis = 1)

fig, ax3 = plt.subplots(1, 1, dpi=300)
ax3.plot(r, Sigmad_nat, '--',color = 'grey',  alpha = 0.5, label = r'$\Sigma_{\mathrm{d,vanilla}}$' )
ax3.plot(r, Sigmad,color = 'green',  label = r'$\Sigma_{\mathrm{d, update}}$')
ax3.grid(True, which = 'both', alpha = 0.2)
ax3.set_xscale('log')
#ax3.set_yscale('log')
ax3.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax3.set_ylabel(r'$\Sigma_{\mathrm{d}}\ [g\ cm^{-2}]$')
ax3.set_xlim(1,900)
ax3.set_title(r'Dust surface density after $2.512\times 10^{4}$years')
ax3.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/Sigmad_data0015.png', dpi=300, bbox_inches="tight")
plt.show

#Sigmag = np.sum(Sigmag, axis = 1)
#Sigmag_nat = np.sum(Sigmag_nat, axis = 1)

fig, ax4 = plt.subplots(1, 1, dpi=300)
ax4.plot(r, Sigmag_nat, '--',color = 'grey',  alpha = 0.5, label = r'$\Sigma_{\mathrm{g,vanilla}}$' )
ax4.plot(r, Sigmag,color = 'green',  label = r'$\Sigma_{\mathrm{g, update}}$')
ax4.grid(True, which = 'both', alpha = 0.2)
ax4.set_xscale('log')
#ax4.set_yscale('log')
ax4.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax4.set_ylabel(r'$\Sigma_{\mathrm{g}}\ [g\ cm^{-2}]$')
ax4.set_xlim(1,900)
ax4.set_title(r'Gas surface density after $2.512\times 10^{4}$years')
ax4.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/Sigmag_data0015.png', dpi=300, bbox_inches="tight")
plt.show

#dust to gas ratio
fig, ax5 = plt.subplots(1, 1, dpi=300)
ax5.plot(r, eps_nat, '--',color = 'grey',  alpha = 0.5, label = r'$\epsilon_{\mathrm{vanilla}}$' )
ax5.plot(r, eps,color = 'green',  label = r'$\epsilon_{\mathrm{update}}$')
ax5.grid(True, which = 'both', alpha = 0.2)
ax5.set_xscale('log')
ax5.set_yscale('log')
ax5.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax5.set_ylabel(r'$\epsilon$')
ax5.set_xlim(1,900)
ax5.set_title(r'Dust to gas ratio after $2.512\times 10^{4}$years')
ax5.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/epsilon_data0015.png', dpi=300, bbox_inches="tight")
plt.show

#relative velocities
v_brown = np.sum(v_brown, axis = 1)
v_brown = np.sum(v_brown, axis = 1)
v_brown_nat = np.sum(v_brown_nat, axis = 1)
v_brown_nat = np.sum(v_brown_nat, axis = 1)

fig, ax6 = plt.subplots(1, 1, dpi=300)
ax6.plot(r, v_brown_nat, '--',color = 'grey',  alpha = 0.5, label = r'$v_{\mathrm{brown,vanilla}}$' )
ax6.plot(r, v_brown,color = 'green',  label = r'$v_{\mathrm{brown, update}}$')
ax6.grid(True, which = 'both', alpha = 0.2)
ax6.set_xscale('log')
ax6.set_yscale('log')
ax6.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax6.set_ylabel(r'$v_{\mathrm{brown}}\ [cm\ g^{-1}]$')
ax6.set_xlim(1,900)
ax6.set_title(r'Velocity after $2.512\times 10^{4}$years')
ax6.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/v_brown_data0015.png', dpi=300, bbox_inches="tight")
plt.show

v_rel_tot = np.sum(v_rel_tot, axis = 1)
v_rel_tot = np.sum(v_rel_tot, axis = 1)
v_rel_tot_nat = np.sum(v_rel_tot_nat, axis = 1)
v_rel_tot_nat = np.sum(v_rel_tot_nat, axis = 1)

fig, ax7 = plt.subplots(1, 1, dpi=300)
ax7.plot(r, v_rel_tot_nat, '--',color = 'grey',  alpha = 0.5, label = r'$v_{\mathrm{rel, tot, ,vanilla}}$' )
ax7.plot(r, v_rel_tot,color = 'green',  label = r'$v_{\mathrm{rel, tot, update}}$')
ax7.grid(True, which = 'both', alpha = 0.2)
ax7.set_xscale('log')
ax7.set_yscale('log')
ax7.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax7.set_ylabel(r'$v_{\mathrm{rel, tot}}\ [cm\ g^{-1}]$')
ax7.set_xlim(1,900)
ax7.set_title(r'Velocity after $2.512\times 10^{4}$years')
ax7.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/v_rel_tot_data0015.png', dpi=300, bbox_inches="tight")
plt.show

#fragmentation probabilities
p_frag = np.sum(p_frag, axis = 1)
p_frag = np.sum(p_frag, axis = 1)
p_frag_nat = np.sum(p_frag_nat, axis = 1)
p_frag_nat = np.sum(p_frag_nat, axis = 1)

fig, ax8 = plt.subplots(1, 1, dpi=300)
ax8.plot(r, p_frag_nat, '--',color = 'grey',  alpha = 0.5, label = r'$p_{\mathrm{frag, vanilla}}$' )
ax8.plot(r, p_frag,color = 'green',  label = r'$p_{\mathrm{frag, update}}$')
ax8.grid(True, which = 'both', alpha = 0.2)
ax8.set_xscale('log')
#ax8.set_yscale('log')
ax8.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax8.set_ylabel(r'$p_{\mathrm{frag}}$')
ax8.set_xlim(1,900)
ax8.set_title(r'Fragmentation probability $2.512\times 10^{4}$years')
ax8.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/p_frag_data0015.png', dpi=300, bbox_inches="tight")
plt.show

p_stick = np.sum(p_stick, axis = 1)
p_stick = np.sum(p_stick, axis = 1)
p_stick_nat = np.sum(p_stick_nat, axis = 1)
p_stick_nat = np.sum(p_stick_nat, axis = 1)

fig, ax9 = plt.subplots(1, 1, dpi=300)
ax9.plot(r, p_stick_nat, '--',color = 'grey',  alpha = 0.5, label = r'$p_{\mathrm{stick ,vanilla}}$' )
ax9.plot(r, p_stick, color = 'green',  label = r'$p_{\mathrm{stick, update}}$')
ax9.grid(True, which = 'both', alpha = 0.2)
ax9.set_xscale('log')
#ax9.set_yscale('log')
ax9.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax9.set_ylabel(r'$p_{\mathrm{stick}}$')
ax9.set_xlim(1,900)
ax9.set_title(r'Sticking probability $2.512\times 10^{4}$years')
ax9.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/p_stick_data0015.png', dpi=300, bbox_inches="tight")
plt.show

In [ ]:
print('#########################################################')
print('modified dustPy, 1e5 yr')
print('#########################################################')


with h5py.File('data/data0021.hdf5','r') as f:
    #print(list(f.keys()))


    r = f['grid/r']
    r = np.asarray(r/apc.au.cgs.value)
    #print(f'r [au]:{r}')


    Sigmad = f['dust/Sigma']
    Sigmad = np.asarray(Sigmad)
    #print(f'Sigma_d [cm^2 g^-1]:{Sigmad}')
    Sigmag = f['gas/Sigma']
    Sigmag = np.asarray(Sigmag)
    #print(f'Sigma_g [cm^2 g^-1]:{Sigmag}')


    Flux_adv = f['dust/Fi/adv']
    Flux_adv = np.asarray(Flux_adv)
    #print(f'F_adv:{Flux_adv}')
    Flux_diff = f['dust/Fi/diff']
    Flux_diff = np.asarray(Flux_diff)
    #print(f'F_diff:{Flux_diff}')
    Flux_tot = f['dust/Fi/tot']
    Flux_tot = np.asarray(Flux_tot)
    #print(f'F_tot:{Flux_tot}')

    v_brown = f['dust/v/rel/brown']
    v_brown = np.asarray(v_brown)
    #print(f'v_brown:{v_brown}')
    v_rel_tot = f['dust/v/rel/tot']
    v_rel_tot = np.asarray(v_rel_tot)
    #print(f'v_rel_tot:{v_rel_tot}')

    eps = f['dust/eps']
    eps = np.asarray(eps)
    #print(f'eps:{eps}')

    coag = f['dust/S/coag']
    coag = np.asarray(coag)
    #print(f'coagulation: {coag}')

    p_frag = f['dust/p/frag']
    p_stick = f['dust/p/stick']
    p_frag = np.asarray(p_frag)
    p_stick = np.asarray(p_stick)


    delta_rad = f['dust/delta/rad']
    delta_rad = np.asarray(delta_rad)
    #print(f'Delta_rad:{delta_rad}')
    delta_turb = f['dust/delta/turb']
    delta_turb = np.asarray(delta_turb)
    #print(f'Delta_turb:{delta_turb}')
    delta_vert = f['dust/delta/vert']
    delta_vert = np.asarray(delta_vert)
    #print(f'Delta_vert:{delta_vert}')

print('#########################################################')
print('vanilla dustPy, 1e5 yr')
print('#########################################################')

with h5py.File('data/data_nat/data0010.hdf5','r') as f:
    #print(list(f.keys()))


    r_nat = f['grid/r']
    r_nat = np.asarray(r_nat/apc.au.cgs.value)
    #print(f'r [au]:{r_nat}')


    Sigmad_nat = f['dust/Sigma']
    Sigmad_nat = np.asarray(Sigmad_nat)
    #print(f'Sigma_d [cm^2 g^-1]:{Sigmad_nat}')
    Sigmag_nat = f['gas/Sigma']
    Sigmag_nat = np.asarray(Sigmag_nat)
    #print(f'Sigma_g [cm^2 g^-1]:{Sigmag_nat}')


    Flux_adv_nat = f['dust/Fi/adv']
    Flux_adv_nat = np.asarray(Flux_adv_nat)
    #print(f'F_adv:{Flux_adv_nat}')
    Flux_diff_nat = f['dust/Fi/diff']
    Flux_diff_nat = np.asarray(Flux_diff_nat)
    #print(f'F_diff:{Flux_diff_nat}')
    Flux_tot_nat = f['dust/Fi/tot']
    Flux_tot_nat = np.asarray(Flux_tot_nat)
    #print(f'F_tot:{Flux_tot_nat}')

    v_brown_nat = f['dust/v/rel/brown']
    v_brown_nat = np.asarray(v_brown_nat)
    #print(f'v_brown:{v_brown_nat}')
    v_rel_tot_nat = f['dust/v/rel/tot']
    v_rel_tot_nat = np.asarray(v_rel_tot_nat)
    #print(f'v_rel_tot:{v_rel_tot_nat}')

    eps_nat = f['dust/eps']
    eps_nat = np.asarray(eps_nat)
    #print(f'eps:{eps_nat}')

    coag_nat = f['dust/S/coag']
    coag_nat = np.asarray(coag_nat)
    #print(f'coagulation: {coag_nat}')

    p_frag_nat = f['dust/p/frag']
    p_stick_nat = f['dust/p/stick']
    p_frag_nat = np.asarray(p_frag_nat)
    p_stick_nat = np.asarray(p_stick_nat)


    delta_rad_nat = f['dust/delta/rad']
    delta_rad_nat = np.asarray(delta_rad_nat)
    #print(f'Delta_rad:{delta_rad_nat}')
    delta_turb_nat = f['dust/delta/turb']
    delta_turb_nat = np.asarray(delta_turb_nat)
    #print(f'Delta_turb:{delta_turb_nat}')
    delta_vert_nat = f['dust/delta/vert']
    delta_vert_nat = np.asarray(delta_vert_nat)
    #print(f'Delta_vert:{delta_vert_nat}')
      
print('#########################################################')
print('comparison , 1e5 yr')
print('#########################################################')

delta_Sigmad = Sigmad - Sigmad_nat
print(f'change in Sigma_d:{delta_Sigmad}')
delta_Sigmag = Sigmag - Sigmag_nat
print(f'change in Sigma_g:{delta_Sigmag}')

delta_flux_adv = Flux_adv - Flux_adv_nat
print(f'change in F_adv: {delta_flux_adv}')
delta_flux_diff = Flux_diff - Flux_diff_nat
print(f'change in F_diff: {delta_flux_diff}')
delta_flux_tot = Flux_tot - Flux_tot_nat
print(f'change in F_tot: {delta_flux_tot}')

delta_v_brown = v_brown - v_brown_nat
print(f'change in v_brown:{delta_v_brown}')
delta_v_rel_tot = v_rel_tot - v_rel_tot_nat
print(f'change in v_rel_tot:{delta_v_rel_tot}')

delta_eps = eps - eps_nat
print(f'change in epsilon:{delta_eps}')

delta_coag = coag - coag_nat
print(f'change in coagulation:{delta_coag}')


with h5py.File('data/data0021.hdf5','r') as f:
    ri = np.asarray(f['grid/ri'])
    ri = ri / apc.au.cgs.value

Flux_adv = np.sum(Flux_adv, axis = 1)
Flux_adv_nat = np.sum(Flux_adv_nat, axis = 1)


fig, ax0 = plt.subplots(1, 1, dpi=300)
ax0.plot(ri, Flux_adv_nat, '--',color = 'grey',  alpha = 0.5, label = r'$F_{\mathrm{adv,vanilla}}$' )
ax0.plot(ri, Flux_adv,color = 'green',  label = r'$F_{\mathrm{adv, update}}$')
ax0.grid(True, which = 'both', alpha = 0.2)
ax0.set_xscale('log')
#ax0.set_yscale('log')
ax0.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax0.set_ylabel(r'$F_{\mathrm{adv}}$')
ax0.set_xlim(1,900)
ax0.set_title(r'Advection flux after $10^{5}$years')
ax0.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/flux_adv_data0021.png', dpi=300, bbox_inches="tight")
plt.show

Flux_diff = np.sum(Flux_diff, axis = 1)
Flux_diff_nat = np.sum(Flux_diff_nat, axis = 1)

fig, ax1 = plt.subplots(1, 1, dpi=300)
ax1.plot(ri, Flux_diff_nat, '--',color = 'grey',  alpha = 0.5, label = r'$F_{\mathrm{diff,vanilla}}$' )
ax1.plot(ri, Flux_diff,color = 'green',  label = r'$F_{\mathrm{diff, update}}$')
ax1.grid(True, which = 'both', alpha = 0.2)
ax1.set_xscale('log')
#ax1.set_yscale('log')
ax1.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax1.set_ylabel(r'$F_{\mathrm{diff}}$')
ax1.set_xlim(1,900)
ax1.set_title(r'Diffusion flux after $10^{5}$years')
ax1.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/flux_diff_data0021.png', dpi=300, bbox_inches="tight")
plt.show

Flux_tot = np.sum(Flux_tot, axis = 1)
Flux_tot_nat = np.sum(Flux_tot_nat, axis = 1)

fig, ax2 = plt.subplots(1, 1, dpi=300)
ax2.plot(ri, Flux_tot_nat, '--',color = 'grey',  alpha = 0.5, label = r'$F_{\mathrm{tot,vanilla}}$' )
ax2.plot(ri, Flux_tot,color = 'green',  label = r'$F_{\mathrm{tot, update}}$')
ax2.grid(True, which = 'both', alpha = 0.2)
ax2.set_xscale('log')
#ax2.set_yscale('log')
ax2.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax2.set_ylabel(r'$F_{\mathrm{tot}}$')
ax2.set_xlim(1,900)
ax2.set_title(r'Total flux after $10^{5}$years')
ax2.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/flux_tot_data0021.png', dpi=300, bbox_inches="tight")
plt.show

#surface denisities
Sigmad = np.sum(Sigmad, axis = 1)
Sigmad_nat = np.sum(Sigmad_nat, axis = 1)

fig, ax3 = plt.subplots(1, 1, dpi=300)
ax3.plot(r, Sigmad_nat, '--',color = 'grey',  alpha = 0.5, label = r'$\Sigma_{\mathrm{d,vanilla}}$' )
ax3.plot(r, Sigmad,color = 'green',  label = r'$\Sigma_{\mathrm{d, update}}$')
ax3.grid(True, which = 'both', alpha = 0.2)
ax3.set_xscale('log')
#ax3.set_yscale('log')
ax3.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax3.set_ylabel(r'$\Sigma_{\mathrm{d}}\ [g\ cm^{-2}]$')
ax3.set_xlim(1,900)
ax3.set_title(r'Dust surface density after $10^{5}$years')
ax3.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/Sigmad_data0021.png', dpi=300, bbox_inches="tight")
plt.show

#Sigmag = np.sum(Sigmag, axis = 1)
#Sigmag_nat = np.sum(Sigmag_nat, axis = 1)

fig, ax4 = plt.subplots(1, 1, dpi=300)
ax4.plot(r, Sigmag_nat, '--',color = 'grey',  alpha = 0.5, label = r'$\Sigma_{\mathrm{g,vanilla}}$' )
ax4.plot(r, Sigmag,color = 'green',  label = r'$\Sigma_{\mathrm{g, update}}$')
ax4.grid(True, which = 'both', alpha = 0.2)
ax4.set_xscale('log')
#ax4.set_yscale('log')
ax4.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax4.set_ylabel(r'$\Sigma_{\mathrm{g}}\ [g\ cm^{-2}]$')
ax4.set_xlim(1,900)
ax4.set_title(r'Gas surface density after $10^{5}$years')
ax4.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/Sigmag_data0021.png', dpi=300, bbox_inches="tight")
plt.show

#dust to gas ratio
fig, ax5 = plt.subplots(1, 1, dpi=300)
ax5.plot(r, eps_nat, '--',color = 'grey',  alpha = 0.5, label = r'$\epsilon_{\mathrm{vanilla}}$' )
ax5.plot(r, eps,color = 'green',  label = r'$\epsilon_{\mathrm{update}}$')
ax5.grid(True, which = 'both', alpha = 0.2)
ax5.set_xscale('log')
ax5.set_yscale('log')
ax5.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax5.set_ylabel(r'$\epsilon$')
ax5.set_xlim(1,900)
ax5.set_title(r'Dust to gas ratio after $10^{5}$years')
ax5.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/epsilon_data0021.png', dpi=300, bbox_inches="tight")
plt.show

#relative velocities
v_brown = np.sum(v_brown, axis = 1)
v_brown = np.sum(v_brown, axis = 1)
v_brown_nat = np.sum(v_brown_nat, axis = 1)
v_brown_nat = np.sum(v_brown_nat, axis = 1)

fig, ax6 = plt.subplots(1, 1, dpi=300)
ax6.plot(r, v_brown_nat, '--',color = 'grey',  alpha = 0.5, label = r'$v_{\mathrm{brown,vanilla}}$' )
ax6.plot(r, v_brown,color = 'green',  label = r'$v_{\mathrm{brown, update}}$')
ax6.grid(True, which = 'both', alpha = 0.2)
ax6.set_xscale('log')
ax6.set_yscale('log')
ax6.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax6.set_ylabel(r'$v_{\mathrm{brown}}\ [cm\ g^{-1}]$')
ax6.set_xlim(1,900)
ax6.set_title(r'Velocity after $10^{5}$years')
ax6.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/v_brown_data0021.png', dpi=300, bbox_inches="tight")
plt.show

v_rel_tot = np.sum(v_rel_tot, axis = 1)
v_rel_tot = np.sum(v_rel_tot, axis = 1)
v_rel_tot_nat = np.sum(v_rel_tot_nat, axis = 1)
v_rel_tot_nat = np.sum(v_rel_tot_nat, axis = 1)

fig, ax7 = plt.subplots(1, 1, dpi=300)
ax7.plot(r, v_rel_tot_nat, '--',color = 'grey',  alpha = 0.5, label = r'$v_{\mathrm{rel, tot, ,vanilla}}$' )
ax7.plot(r, v_rel_tot,color = 'green',  label = r'$v_{\mathrm{rel, tot, update}}$')
ax7.grid(True, which = 'both', alpha = 0.2)
ax7.set_xscale('log')
ax7.set_yscale('log')
ax7.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax7.set_ylabel(r'$v_{\mathrm{rel, tot}}\ [cm\ g^{-1}]$')
ax7.set_xlim(1,900)
ax7.set_title(r'Velocity after $10^{5}$years')
ax7.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/v_rel_tot_data0021.png', dpi=300, bbox_inches="tight")
plt.show

#fragmentation probabilities
p_frag = np.sum(p_frag, axis = 1)
p_frag = np.sum(p_frag, axis = 1)
p_frag_nat = np.sum(p_frag_nat, axis = 1)
p_frag_nat = np.sum(p_frag_nat, axis = 1)

fig, ax8 = plt.subplots(1, 1, dpi=300)
ax8.plot(r, p_frag_nat, '--',color = 'grey',  alpha = 0.5, label = r'$p_{\mathrm{frag ,vanilla}}$' )
ax8.plot(r, p_frag,color = 'green',  label = r'$p_{\mathrm{frag, update}}$')
ax8.grid(True, which = 'both', alpha = 0.2)
ax8.set_xscale('log')
#ax8.set_yscale('log')
ax8.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax8.set_ylabel(r'$p_{\mathrm{frag}}$')
ax8.set_xlim(1,900)
ax8.set_title(r'Fragmentation probability $10^{5}$years')
ax8.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/p_frag_data0021.png', dpi=300, bbox_inches="tight")
plt.show

p_stick = np.sum(p_stick, axis = 1)
p_stick = np.sum(p_stick, axis = 1)
p_stick_nat = np.sum(p_stick_nat, axis = 1)
p_stick_nat = np.sum(p_stick_nat, axis = 1)

fig, ax9 = plt.subplots(1, 1, dpi=300)
ax9.plot(r, p_stick_nat, '--',color = 'grey',  alpha = 0.5, label = r'$p_{\mathrm{stick, vanilla}}$' )
ax9.plot(r, p_stick, color = 'green',  label = r'$p_{\mathrm{stick, update}}$')
ax9.grid(True, which = 'both', alpha = 0.2)
ax9.set_xscale('log')
#ax9.set_yscale('log')
ax9.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax9.set_ylabel(r'$p_{\mathrm{stick}}$')
ax9.set_xlim(1,900)
ax9.set_title(r'Sticking probability $10^{5}$years')
ax9.legend(loc = 'best')

plt.tight_layout()
fig.savefig('pictures/p_stick_data0021.png', dpi=300, bbox_inches="tight")
plt.show
